# Aula 05 - Árvores de Decisão: Churn de Clientes de Internet
# Objetivos:
# - Entender árvores de decisão como modelo de classificação.
# - Treinar um modelo para prever churn (cancelamento de plano).
# - Interpretar as regras aprendidas pela árvore.
# - Discutir overfitting e hiperparâmetros básicos (max_depth).

## Contexto

Você é analista de dados em uma operadora de internet em São Paulo.  
O time de marketing quer identificar quais clientes têm maior risco de **churn** (cancelar o plano) nos próximos meses, para oferecer promoções e planos especiais de retenção.

Vamos treinar uma **árvore de decisão** para classificar clientes em:
- 0 = Não vai churnar (tendência a permanecer)
- 1 = Vai churnar (alto risco de cancelamento)

Seção 1 – Setup e dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set(style="whitegrid")

Seção 2 - Carregando/gerando dados

OBS: criamos dados “realistas” com uma regra por trás, para a árvore tentar descobrir.

In [ ]:
# Gerando um dataset sintético de churn (para exemplo)
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "idade": np.random.randint(18, 80, size=n),
    "tempo_de_casa": np.random.randint(1, 60, size=n),  # em meses
    "valor_mensal": np.random.choice([79.9, 99.9, 129.9, 159.9], size=n),
    "tipo_plano": np.random.choice(["Basico", "Intermediario", "Premium"], size=n),
    "atrasos_pagamento": np.random.poisson(lam=1.5, size=n),
    "usa_app": np.random.choice([0, 1], size=n),
    "atendimentos_suporte": np.random.poisson(lam=0.8, size=n)
})

# Regra "escondida" para gerar churn
prob_churn = (
    0.15
    + 0.25*(df["tempo_de_casa"] < 6)
    + 0.20*(df["atrasos_pagamento"] >= 3)
    + 0.15*(df["atendimentos_suporte"] >= 2)
    + 0.10*(df["valor_mensal"] >= 129.9)
)
prob_churn = np.clip(prob_churn, 0, 0.9)

df["churn"] = np.random.binomial(1, prob_churn)

df.head()

Seção 2 – EDA rápida

In [ ]:
df.info()

In [ ]:
df["churn"].value_counts(normalize=True)

In [ ]:
sns.countplot(x="churn", data=df)
plt.title("Distribuição de Churn (0 = fica, 1 = sai)")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="churn", y="tempo_de_casa", data=df)
plt.title("Tempo de casa x Churn")
plt.show()

Seção 3 – Pré-processamento

- Categóricas → dummies
- Split treino/teste
- Sem normalização (atenção para esse ponto!)

In [ ]:
df_model = df.copy()

df_model = pd.get_dummies(df_model, columns=["tipo_plano"], drop_first=True)

X = df_model.drop("churn", axis=1)
y = df_model["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

- Note: Diferente do KNN, não estamos normalizando os dados.
- Árvores de decisão não se importam com a escala das variáveis,
- Elas só avaliam "cortes" (ex: tempo_de_casa < 10?).

Seção 4 – Treinando a Decision Tree (modelo base)

In [ ]:
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Acurácia (teste): {acc:.3f}")

Matriz de confusão:

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - Árvore de Decisão")
plt.show()

Opcional:

In [ ]:
print(classification_report(y_test, y_pred))

Seção 5 – Interpretando a árvore (visualização simples)

Para não ficar gigante, limitar profundidade na visualização:

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=["Fica", "Sai"],
    filled=True,
    max_depth=3,
    fontsize=8
)
plt.show()

### Perguntas para ler a árvore

- Qual é a primeira pergunta (raiz) que a árvore faz? Faz sentido para o negócio?
- Em qual condição a árvore considera maior risco de churn?
- Você consegue explicar uma regra em linguagem de negócio?  
  Ex.: "Clientes com **pouco tempo de casa** e **muitos atrasos** têm alta chance de churn."

Seção 6 – Overfitting x Underfitting (max_depth)

Mostrar acurácia em treino x teste:

In [ ]:
y_train_pred = clf.predict(X_train)
acc_train = accuracy_score(y_train, y_train_pred)

print(f"Acurácia treino: {acc_train:.3f}")
print(f"Acurácia teste:  {acc:.3f}")

Depois, comparar com árvore “podada”:

In [ ]:
clf2 = DecisionTreeClassifier(max_depth=4, random_state=42)
clf2.fit(X_train, y_train)

y_train_pred2 = clf2.predict(X_train)
y_test_pred2 = clf2.predict(X_test)

print("Árvore com max_depth=4")
print(f"Acurácia treino: {accuracy_score(y_train, y_train_pred2):.3f}")
print(f"Acurácia teste:  {accuracy_score(y_test, y_test_pred2):.3f}")

REFLEXÃO

- O que aconteceu com a acurácia de treino quando limitamos a profundidade?
- E com a acurácia de teste?
- Isso é um exemplo de **overfitting** ou **underfitting**?

Seção 7 – Importância das variáveis

In [ ]:
importances = pd.Series(clf2.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(8,4))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Importância das Features (Árvore max_depth=4)")
plt.show()

Pergunta:

As variáveis mais importantes fazem sentido para explicar "churn" na vida real?

Seção 8 – (Opcional) Random Forest Teaser

Se der tempo, QUE TAL um “sneak peek” de ensemble?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Acurácia Random Forest:", accuracy_score(y_test, y_pred_rf))

EXPLICAÇÃO RÁPIDA

### O que é uma Random Forest?

- É um **conjunto de árvores de decisão**, cada uma treinada com:
  - uma amostra aleatória dos dados,
  - e uma amostra aleatória das variáveis.
- A predição final é o **voto da maioria** das árvores.
- Em geral, é **mais robusta** e generaliza melhor do que uma única árvore.

Seção 9 – Conclusão

## Conclusões da Aula

- Árvores de decisão aprendem **regras de decisão** a partir dos dados (sequência de if/else).
- São modelos **interpretáveis**, ótimos para explicar decisões ao time de negócio.
- Precisam de cuidado com **overfitting** (profundidade muito grande pode "decorar" os dados).
- Random Forest (conjunto de árvores) costuma melhorar a performance e a robustez do modelo.

**Para pensar:**
- Em quais problemas da sua área de trabalho um modelo interpretável seria obrigatório?
- Que outros dados você coletaria para melhorar esse modelo de churn?